# Augmentation end to end

Booster LEAP, Computer Vision, topic 1. Does augmentation really help, and when?

> **First step: make your own copy.** This notebook is read-only (it lives in a public, shared place), so Colab cannot save your changes here and may ask for a GitHub login. Choose **File, Save a copy in Drive** now, then work in your copy. It is saved in your Google Drive under *Colab Notebooks*, where you can edit, run and save it as you like. You can still run the cells without a copy; your changes are then lost when you close the tab.

**How to use this notebook:** run the cells from top to bottom (Runtime, Run all, or Shift+Enter on each cell). It needs no GPU and no downloads: the data is made in the code or ships with scikit-learn. Change one thing at a time and run again; the exercises at the end tell you what to try.

## Stage 1: a model you can read completely

Task: tell vertical bars from horizontal bars on an 8x8 picture. The model stores the *average image* of each class and gives a new image the class whose average is closer.

1. Make the data: a function draws a bar of length 5 and thickness 2.
2. Train with only two pictures, one bar of each kind, both in the middle.
3. Test with 200 bars at random positions.
4. Augment: add copies of the two training pictures shifted by up to 2 columns and 1 row, then compare.

In [ ]:
import random

N = 8

def bar(kind, r, c):
    # a bar of length 5 and thickness 2 on an 8x8 image: "v" is vertical, "h" is horizontal
    img = [[0] * N for _ in range(N)]
    for i in range(5):
        for t in range(2):
            if kind == "v":
                img[r + i][c + t] = 9
            else:
                img[c + t][r + i] = 9
    return img

def flat(x):
    return [v for row in x for v in row]

def shift(x, dx, dy):
    return [[x[i - dy][j - dx] if 0 <= i - dy < N and 0 <= j - dx < N else 0 for j in range(N)] for i in range(N)]

def centroids(data):
    # the "model": the average image of each class
    out = {}
    for k in "vh":
        images = [flat(x) for x, y in data if y == k]
        out[k] = [sum(col) / len(images) for col in zip(*images)]
    return out

def predict(cents, x):
    f = flat(x)
    return min(cents, key=lambda k: sum((a - b) ** 2 for a, b in zip(cents[k], f)))

# training data: only ONE example per class, both in the middle of the picture
train = [(bar("v", 1, 3), "v"), (bar("h", 1, 3), "h")]

# test data: 200 bars at random positions
rng = random.Random(3)
test = []
for _ in range(200):
    k = rng.choice("vh")
    test.append((bar(k, rng.randint(0, 3), rng.randint(1, 5)), k))

def accuracy(cents):
    return sum(predict(cents, x) == y for x, y in test) / len(test)

print("training images:", len(train), " without augmentation, test accuracy:", accuracy(centroids(train)))

# augmentation: add shifted copies (2 columns left or right, 1 row up or down) of every training image
augmented = list(train)
for x, y in train:
    for dx in range(-2, 3):
        for dy in range(-1, 2):
            augmented.append((shift(x, dx, dy), y))
print("training images:", len(augmented), " with shift augmentation, test accuracy:", accuracy(centroids(augmented)))

training images: 2  without augmentation, test accuracy: 0.35
training images: 32  with shift augmentation, test accuracy: 0.58


The accuracy goes from 0.35 to 0.58. It was below 0.5 at first because a bar in a new place can overlap the average picture of the *wrong* class more than that of its own class. The shifted copies spread the average over the places where bars can appear. The model is crude, so the lesson is the direction of the change.

## Stage 2: a small CNN on real digit images

scikit-learn contains 1,797 digit pictures of 8x8 pixels. We keep only **150** for training on purpose: augmentation helps most when data is scarce. We test on two versions of the same test images: unchanged and randomly shifted. (The exact numbers can differ slightly on another machine.)

In [ ]:
import torch
from torch import nn
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

X, y = load_digits(return_X_y=True)                                      # 1,797 digits of 8x8 pixels (ships with scikit-learn)
X = torch.tensor(X / 16.0, dtype=torch.float32).reshape(-1, 1, 8, 8)
y = torch.tensor(y)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
Xtr, ytr = Xtr[:150], ytr[:150]                                          # a SMALL training set: 150 images
print("train", tuple(Xtr.shape), "test", tuple(Xte.shape))

def shifted(x, gen):
    # move every image by -1, 0 or +1 pixels in both directions (wrapping around the border)
    out = torch.zeros_like(x)
    for i in range(len(x)):
        dx = int(torch.randint(-1, 2, (1,), generator=gen))
        dy = int(torch.randint(-1, 2, (1,), generator=gen))
        out[i] = torch.roll(x[i], (dy, dx), (1, 2))
    return out

Xte_shifted = shifted(Xte, torch.Generator().manual_seed(5))              # the same test images, but moved

def make_net():
    return nn.Sequential(nn.Conv2d(1, 8, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                         nn.Conv2d(8, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2), nn.Flatten(), nn.Linear(64, 10))

def train_and_test(augment):
    torch.manual_seed(0)
    net = make_net()
    opt = torch.optim.Adam(net.parameters(), lr=0.01)
    loss_fn = nn.CrossEntropyLoss()
    gen = torch.Generator().manual_seed(1)
    for epoch in range(40):
        net.train()
        order = torch.randperm(len(Xtr))
        for i in range(0, len(Xtr), 32):
            idx = order[i:i + 32]
            xb = Xtr[idx]
            if augment:
                xb = shifted(xb, gen)                                    # new random shifts in every epoch
            opt.zero_grad()
            loss_fn(net(xb), ytr[idx]).backward()
            opt.step()
    net.eval()
    with torch.no_grad():
        return (net(Xte).argmax(1) == yte).float().mean().item(), (net(Xte_shifted).argmax(1) == yte).float().mean().item()

for name, aug in (("no augmentation", False), ("random shifts  ", True)):
    a, b = train_and_test(aug)
    print(f"{name}: accuracy on the test images {a:.3f}, on the same images shifted {b:.3f}")

no augmentation: accuracy on the test images 0.917, on the same images shifted 0.437
random shifts  : accuracy on the test images 0.917, on the same images shifted 0.824


On the unchanged test images the two models are equally good (0.917 and 0.917). On the shifted images the plain model falls to 0.437 while the augmented one keeps 0.824. The gain appears only where the test images contain the changes the training augmentation covered.

## Try it yourself

1. In stage 1, change the shift range (`range(-2, 3)` and `range(-1, 2)`) to `range(-4, 5)` and `range(-2, 3)`. What happens to the accuracy, and why?
2. In stage 1, train with 3 bars per class at different positions instead of 1. Is augmentation still needed?
3. In stage 2, change the number of training images from 150 to 600. Does the benefit of augmentation shrink?
4. In stage 2, add a random horizontal flip to `shifted`. Digits are not mirror-symmetric: does the accuracy go up or down?
5. Write down the recipe: list expected changes, add matching augmentation to training only, test on data with those changes and on clean data, compare with the same seed.